In [11]:
import pandas as pd
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed

from pd_estim_A.models.nig.nig_gibbs_weekly_updated import (
    process_one_firm_nig,
)
from pd_estim_A.data.data_import import (
    load_data, load_ecb_1y_yield,
    fill_liabilities, drop_high_leverage_firms,
    prepare_nig_inputs
)
from pd_estim_A.data.cds_df import get_cds_panel
from pd_estim_A.models.nig.nig_apath import (
    NIGParams,
)

In [12]:
# Paths
print(Path.cwd())
data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"

# Load data and prepare panels
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file= output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,  # recommended if it works
)

df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)

debt_daily = fill_liabilities(bs, df_cal)

ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel consistent with filtered firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()


nig_df, em_cache = prepare_nig_inputs(ret_filt, bs_filt, df_rf, debt_daily=debt_daily_filt, build_em=False)
print(nig_df.head())
print(nig_df.shape)
print(nig_df.describe())

c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
Data has been written to c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\ecb_yc_1y_aaa.xml
[drop_high_leverage_firms] agg=median, threshold=8.0
[drop_high_leverage_firms] firms before: 46 | after: 36
[drop_high_leverage_firms] dropped firms: 10
    gvkey       date             E          isin  \
0  100022 2012-01-03  3.328431e+10  DE0005190003   
1  100080 2012-01-03  4.268705e+10  DE000BAY0017   
2  100312 2012-01-03  1.469717e+09  DE0007030009   
3  100581 2012-01-03  4.935351e+10  FR0000120321   
4  100957 2012-01-03  2.931851e+10  ES0144580Y14   

                        company country_iso         r             L  


In [13]:
# call cds panel and merge
cds = get_cds_panel(
    project_root= Path.cwd() / "..",
    save_csv=False,
    verbose=True,
)
# ensure types
merton = nig_df.copy()
merton["gvkey"] = merton["gvkey"].astype(str)
merton["date"]  = pd.to_datetime(merton["date"])

cds["gvkey"] = cds["gvkey"].astype(str)
cds["date"]  = pd.to_datetime(cds["date"])

# keep only firms that exist in BOTH (drop firms with no CDS)
common_gv = sorted(set(merton["gvkey"].unique()) & set(cds["gvkey"].unique()))
merton = merton[merton["gvkey"].isin(common_gv)].copy()
cds = cds[cds["gvkey"].isin(common_gv)].copy()

# also drop CDS rows whose dates are outside merged's date range
dmin, dmax = merton["date"].min(), merton["date"].max()
cds = cds[(cds["date"] >= dmin) & (cds["date"] <= dmax)].copy()

# merge-asof onto merged's dates (direction='backward')
merton = merton.sort_values(["date", "gvkey"]).reset_index(drop=True)
cds    = cds.sort_values(["date", "gvkey"]).reset_index(drop=True)

merged_cds = pd.merge_asof(
    merton,
    cds,
    on="date",
    by="gvkey",
    direction="backward",
    allow_exact_matches=True,
)

# drop rows where CDS still missing
nig_df = merged_cds.dropna(subset=["cds"]).reset_index(drop=True)

print("firms after intersection:", nig_df["gvkey"].nunique())
print("rows after merge:", len(nig_df))
print("date range:", nig_df["date"].min(), "→", nig_df["date"].max())

[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
[get_cds_panel] sheets read: 22 | rows parsed: 67015 | unmapped sheets: 0
firms after intersection: 21
rows after merge: 65569
date range: 2014-01-01 00:00:00 → 2025-12-19 00:00:00


In [14]:
# Cell 2 — rolling configuration

TRAIN_YEARS = 2
STEP_FREQ = "QE"
WEEK_ENDING = "W-FRI"

T_INV = 1.0                  # 1Y maturity used in inversion
HORIZON_WEEKS = 52.0         # 1Y PD horizon on weekly scale

DATA_END = pd.Timestamp("2024-12-31")   # keep equal to Merton if you want clean comparison
LAST_TRAIN_END = DATA_END - pd.offsets.QuarterEnd(1)

MIN_DAILY_ROWS = 10
MIN_WEEKLY_RETURNS = 60      # require enough weekly implied asset returns in each 2Y window

P0 = NIGParams(alpha=15.0, beta=-3.0, delta=0.20, mu=0.00)

INVERT_U = 120.0
INVERT_N = 2000

EM_MAX_ITER = 80
EM_TOL = 1e-6

MAX_FIRMS = 1
MAX_WINDOWS = 1

In [15]:
# Cell 3 — panel preparation
# Assumes nig_df already exists and has at least:
# gvkey, date, E, L, r
# plus optionally company, country_iso, etc.

panel = nig_df.copy()
panel["gvkey"] = panel["gvkey"].astype(str)
panel["date"] = pd.to_datetime(panel["date"])

needed_cols = ["gvkey", "date", "company", "E", "L", "r"]
panel = panel[[c for c in needed_cols if c in panel.columns]].copy()

for c in ["E", "L", "r"]:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel = (
    panel.dropna(subset=["gvkey", "date", "E", "L", "r"])
         .query("E > 0 and L > 0")
         .sort_values(["gvkey", "date"])
         .reset_index(drop=True)
)

firm_daily = {}
for gvkey, g in panel.groupby("gvkey", sort=False):
    g = g.sort_values("date").groupby("date", as_index=False).last()
    firm_daily[gvkey] = g.set_index("date")

gvkeys_all = sorted(firm_daily.keys())
if MAX_FIRMS is not None:
    gvkeys_all = gvkeys_all[:int(MAX_FIRMS)]

print("Firms loaded:", len(firm_daily), "| Firms in run:", len(gvkeys_all))
print("Panel date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("LAST_TRAIN_END:", LAST_TRAIN_END.date(), "| DATA_END:", DATA_END.date())
display(panel.head())

Firms loaded: 21 | Firms in run: 1
Panel date range: 2014-01-01 to 2025-12-19
LAST_TRAIN_END: 2024-09-30 | DATA_END: 2024-12-31


,gvkey,date,company,E,L,r
0,100022,2014-01-01,BAYERISCHE MOTOREN WERKE AKT,5.130203e+10,1.014480e+11,0.000942
1,100022,2014-01-02,BAYERISCHE MOTOREN WERKE AKT,5.029068e+10,1.014480e+11,0.001069
2,100022,2014-01-03,BAYERISCHE MOTOREN WERKE AKT,5.055556e+10,1.014480e+11,0.000991
3,100022,2014-01-06,BAYERISCHE MOTOREN WERKE AKT,4.995958e+10,1.014480e+11,0.001006
4,100022,2014-01-07,BAYERISCHE MOTOREN WERKE AKT,5.029670e+10,1.014480e+11,0.001057


In [16]:
# Cell 4 — build the rolling window schedule

global_min_date = panel["date"].min()
earliest_end = global_min_date + pd.DateOffset(years=TRAIN_YEARS) - pd.Timedelta(days=1)

train_ends = pd.date_range(start=earliest_end, end=LAST_TRAIN_END, freq=STEP_FREQ)
train_ends = pd.to_datetime(train_ends)

if MAX_WINDOWS is not None:
    train_ends = train_ends[:int(MAX_WINDOWS)]

windows = []
for train_end in train_ends:
    train_start = train_end - pd.DateOffset(years=TRAIN_YEARS) + pd.Timedelta(days=1)
    oos_start = train_end + pd.Timedelta(days=1)
    oos_end = train_end + pd.offsets.QuarterEnd(1)

    windows.append(
        {
            "train_start": pd.Timestamp(train_start),
            "train_end": pd.Timestamp(train_end),
            "oos_start": pd.Timestamp(oos_start),
            "oos_end": pd.Timestamp(oos_end),
        }
    )

windows_df = pd.DataFrame(windows)
display(windows_df.head())
display(windows_df.tail())
print("n_windows:", len(windows_df))

,train_start,train_end,oos_start,oos_end
0,2014-01-01,2015-12-31,2016-01-01,2016-03-31


,train_start,train_end,oos_start,oos_end
0,2014-01-01,2015-12-31,2016-01-01,2016-03-31


n_windows: 1


In [17]:
# Cell 7 — Gibbs configuration

# Gibbs settings
GIBBS_MAX_ITER = 100
GIBBS_BURN_IN = 20
GIBBS_THIN = 2

# Weekly/NIG settings
TAU_YEARS = 1.0
WEEKS_PER_YEAR = 52
MIN_WEEKLY_OBS = None
MIN_WEEKLY_RETURNS = 2

# Inversion grid
INVERT_U = 100.0
INVERT_N = 1000

# Reproducibility
BASE_SEED = 12345

In [18]:
# Cell 8 — load EM parameter source for the Bayesian NIG sampler

output_path = Path.cwd() / ".." / "data" / "derived"

em_params_df = pd.read_csv(output_path / "NIG_weekly_rolling.csv")

# Standardize columns for the Gibbs helper
em_params_df["gvkey"] = em_params_df["gvkey"].astype(str)
em_params_df["date"] = pd.to_datetime(em_params_df["date"], errors="coerce")

if "train_end_date" in em_params_df.columns:
    em_params_df["train_end"] = pd.to_datetime(em_params_df["train_end_date"], errors="coerce")
elif "train_end" in em_params_df.columns:
    em_params_df["train_end"] = pd.to_datetime(em_params_df["train_end"], errors="coerce")
else:
    raise ValueError("EM source must contain either 'train_end_date' or 'train_end'.")

# Keep only training-end rows if the file contains both train-end and OOS rows
if "training_end" in em_params_df.columns:
    em_params_df = em_params_df[em_params_df["training_end"] == 1].copy()

required_cols = ["gvkey", "train_end", "alpha", "beta", "delta", "mu"]
missing = [c for c in required_cols if c not in em_params_df.columns]
if missing:
    raise ValueError(f"EM parameter source is missing required columns: {missing}")

em_params_df = (
    em_params_df[required_cols]
    .dropna(subset=["gvkey", "train_end", "alpha", "beta", "delta", "mu"])
    .sort_values(["gvkey", "train_end"])
    .drop_duplicates(subset=["gvkey", "train_end"], keep="last")
    .reset_index(drop=True)
)

print("EM parameter rows:", len(em_params_df))
print("EM firms:", em_params_df["gvkey"].nunique())
display(em_params_df.head())

EM parameter rows: 756
EM firms: 21


,gvkey,train_end,alpha,beta,delta,mu
0,100022,2015-12-31,3.0,0.003605,0.000630,0.001448
1,100022,2016-03-31,3.0,0.000135,0.000877,0.001113
2,100022,2016-06-30,3.0,0.000206,0.000895,0.000573
3,100022,2016-09-30,3.0,0.000180,0.000872,0.001263
4,100022,2016-12-31,3.0,0.000192,0.000868,0.001595


In [19]:
# Cell 9 — prior overrides (leave as None to use default EM-centered priors from the .py file)

PRIOR_B0 = None
PRIOR_B0_COV = None
PRIOR_HYPER = None

In [20]:
# Cell 10 — rolling Bayesian NIG estimation (parallelized across firms)

gvkeys_run = gvkeys_all[:int(MAX_FIRMS)] if MAX_FIRMS is not None else gvkeys_all
windows_run = windows[:int(MAX_WINDOWS)] if MAX_WINDOWS is not None else windows

print(f"Running {len(gvkeys_run)} firms across {len(windows_run)} windows")
print("Parallelization: across firms | sequential within each firm")

# One deterministic seed per firm
seed_map = {gv: BASE_SEED + i for i, gv in enumerate(gvkeys_run)}

parallel_results = Parallel(
    n_jobs=-1,
    backend="loky",
    verbose=10,
)(
    delayed(process_one_firm_nig)(
        firm_daily[gvkey],
        windows=windows_run,
        em_params_source=em_params_df,
        gvkey=gvkey,
        date_col="date",
        week_ending=WEEK_ENDING,
        min_daily_rows=MIN_DAILY_ROWS,
        min_weekly_obs=MIN_WEEKLY_OBS,
        min_weekly_returns=MIN_WEEKLY_RETURNS,
        E_col="E",
        L_col="L",
        r_col="r",
        tau_years=TAU_YEARS,
        weeks_per_year=WEEKS_PER_YEAR,
        max_iter=GIBBS_MAX_ITER,
        burn_in=GIBBS_BURN_IN,
        thin=GIBBS_THIN,
        U=INVERT_U,
        n_int=INVERT_N,
        prior_b0=PRIOR_B0,
        prior_B0=PRIOR_B0_COV,
        prior_hyper=PRIOR_HYPER,
        rng=np.random.default_rng(seed_map[gvkey]),
    )
    for gvkey in gvkeys_run
)

summary_parts = []
weekly_is_parts = []
posterior_results_all = []

for summary_df_i, weekly_is_df_i, posterior_results_i in parallel_results:
    if summary_df_i is not None and not summary_df_i.empty:
        summary_parts.append(summary_df_i)

    if weekly_is_df_i is not None and not weekly_is_df_i.empty:
        weekly_is_parts.append(weekly_is_df_i)

    if posterior_results_i is not None and len(posterior_results_i):
        posterior_results_all.extend(posterior_results_i)

roll_summary_nig_bayes_df = (
    pd.concat(summary_parts, ignore_index=True)
      .sort_values(["train_end", "gvkey"])
      .reset_index(drop=True)
) if len(summary_parts) else pd.DataFrame()

roll_weekly_nig_bayes_is_df = (
    pd.concat(weekly_is_parts, ignore_index=True)
      .sort_values(["train_end", "gvkey", "date"])
      .reset_index(drop=True)
) if len(weekly_is_parts) else pd.DataFrame()

print("roll_summary_nig_bayes_df shape:", roll_summary_nig_bayes_df.shape)
print("roll_weekly_nig_bayes_is_df shape:", roll_weekly_nig_bayes_is_df.shape)
print("n posterior window objects:", len(posterior_results_all))

display(roll_summary_nig_bayes_df.head())
display(roll_weekly_nig_bayes_is_df.head())

Running 1 firms across 1 windows
Parallelization: across firms | sequential within each firm


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


roll_summary_nig_bayes_df shape: (1, 33)
roll_weekly_nig_bayes_is_df shape: (0, 0)
n posterior window objects: 0


[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    7.6s


,gvkey,train_start,train_end,oos_start,oos_end,train_start_req,train_end_req,train_start_used_daily,train_end_used_daily,train_start_used_weekly,...,delta_post_median,mu_post_median,A_last_mean,A_last_median,n_daily_train,n_weekly_train,n_weekly_returns,n_keep_requested,n_keep_actual,n_reject
0,100022,2014-01-01,2015-12-31,2016-01-01,2016-03-31,2014-01-01,2015-12-31,2014-01-01,2015-12-31,2014-01-03,...,NaN,NaN,NaN,NaN,522,105,104,40,0,100


""


In [21]:
display(
    roll_summary_nig_bayes_df[
        [
            "gvkey", "train_end", "ok", "msg",
            "n_daily_train", "n_weekly_train", "n_weekly_returns",
            "n_keep_requested", "n_keep_actual", "n_reject"
        ]
    ]
)

,gvkey,train_end,ok,msg,n_daily_train,n_weekly_train,n_weekly_returns,n_keep_requested,n_keep_actual,n_reject
0,100022,2015-12-31,False,no_kept_draws,522,105,104,40,0,100


In [22]:
# Cell 11 — quick diagnostics

if not roll_summary_nig_bayes_df.empty:
    print("Unique train_end windows:", roll_summary_nig_bayes_df["train_end"].nunique())
    print("Share ok:", roll_summary_nig_bayes_df["ok"].mean())

    ok_df = roll_summary_nig_bayes_df.loc[roll_summary_nig_bayes_df["ok"] == True].copy()

    if not ok_df.empty:
        print("\nPosterior parameter summaries (ok windows only):")
        display(
            ok_df[
                [
                    "alpha_post_mean", "beta_post_mean", "delta_post_mean", "mu_post_mean",
                    "alpha_post_median", "beta_post_median", "delta_post_median", "mu_post_median",
                    "n_keep_actual", "n_reject"
                ]
            ].describe()
        )

        g0 = ok_df["gvkey"].astype(str).unique()
        if len(g0):
            example = g0[0]
            display(
                ok_df.loc[ok_df["gvkey"] == example, [
                    "gvkey", "train_end",
                    "em_alpha", "em_beta", "em_delta", "em_mu",
                    "alpha_post_mean", "beta_post_mean", "delta_post_mean", "mu_post_mean",
                    "n_keep_actual", "n_reject"
                ]].head(15)
            )

if not roll_weekly_nig_bayes_is_df.empty:
    ok_gv = roll_weekly_nig_bayes_is_df["gvkey"].astype(str).unique()
    if len(ok_gv):
        example = ok_gv[0]
        display(
            roll_weekly_nig_bayes_is_df.loc[
                roll_weekly_nig_bayes_is_df["gvkey"] == example,
                [
                    "gvkey", "train_end", "date",
                    "A_mean", "A_median", "A_q05", "A_q95",
                    "theta_mean", "theta_median",
                    "alpha_mean", "beta_mean", "delta_mean", "mu_mean"
                ]
            ].head(30)
        )

Unique train_end windows: 1
Share ok: 0.0


In [23]:
# Cell 12 — build a training-end posterior panel safely

roll_summary_nig_bayes_df = roll_summary_nig_bayes_df.copy()
roll_weekly_nig_bayes_is_df = roll_weekly_nig_bayes_is_df.copy()

for df_ in [roll_summary_nig_bayes_df, roll_weekly_nig_bayes_is_df]:
    if "gvkey" in df_.columns:
        df_["gvkey"] = df_["gvkey"].astype(str)
    if "date" in df_.columns:
        df_["date"] = pd.to_datetime(df_["date"], errors="coerce")
    if "train_end" in df_.columns:
        df_["train_end"] = pd.to_datetime(df_["train_end"], errors="coerce")

ok_summary = roll_summary_nig_bayes_df.loc[roll_summary_nig_bayes_df["ok"] == True].copy()

if roll_weekly_nig_bayes_is_df.empty or ("gvkey" not in roll_weekly_nig_bayes_is_df.columns):
    print("No successful weekly posterior panels were produced.")
    final_nig_bayes_train_df = ok_summary.copy()

    # add empty columns so the output schema is stable
    for c in ["date", "A_0_median", "A_used_median", "L_used"]:
        if c not in final_nig_bayes_train_df.columns:
            final_nig_bayes_train_df[c] = np.nan

    keep_cols = [
        "gvkey", "train_end",
        "date",
        "em_alpha", "em_beta", "em_delta", "em_mu",
        "alpha_post_mean", "beta_post_mean", "delta_post_mean", "mu_post_mean",
        "alpha_post_median", "beta_post_median", "delta_post_median", "mu_post_median",
        "A_0_median", "A_used_median", "L_used",
        "n_keep_requested", "n_keep_actual", "n_reject", "msg"
    ]
    keep_cols = [c for c in keep_cols if c in final_nig_bayes_train_df.columns]
    final_nig_bayes_train_df = final_nig_bayes_train_df[keep_cols].copy()

else:
    is_sorted = roll_weekly_nig_bayes_is_df.sort_values(["gvkey", "train_end", "date"]).copy()

    is_first = (
        is_sorted.groupby(["gvkey", "train_end"], as_index=False)
        .head(1)[["gvkey", "train_end", "A_median"]]
        .rename(columns={"A_median": "A_0_median"})
    )

    is_last = (
        is_sorted.groupby(["gvkey", "train_end"], as_index=False)
        .tail(1)[["gvkey", "train_end", "date", "A_median", "L"]]
        .rename(columns={"A_median": "A_used_median", "L": "L_used"})
    )

    final_nig_bayes_train_df = (
        ok_summary
        .merge(is_first, on=["gvkey", "train_end"], how="left")
        .merge(is_last, on=["gvkey", "train_end"], how="left")
    )

    final_nig_bayes_train_df = final_nig_bayes_train_df[
        [
            "gvkey",
            "train_end",
            "date",
            "em_alpha", "em_beta", "em_delta", "em_mu",
            "alpha_post_mean", "beta_post_mean", "delta_post_mean", "mu_post_mean",
            "alpha_post_median", "beta_post_median", "delta_post_median", "mu_post_median",
            "A_0_median", "A_used_median", "L_used",
            "n_keep_requested", "n_keep_actual", "n_reject", "msg"
        ]
    ].copy()

display(final_nig_bayes_train_df.head(20))
print(final_nig_bayes_train_df.shape)

No successful weekly posterior panels were produced.


,gvkey,train_end,date,em_alpha,em_beta,em_delta,em_mu,alpha_post_mean,beta_post_mean,delta_post_mean,...,beta_post_median,delta_post_median,mu_post_median,A_0_median,A_used_median,L_used,n_keep_requested,n_keep_actual,n_reject,msg


(0, 22)


In [24]:
# Cell 13 — save outputs

output_path = Path.cwd() / ".." / "data" / "derived"
output_path.mkdir(parents=True, exist_ok=True)

roll_summary_nig_bayes_df.to_csv(output_path / "nig_bayes_roll_summary.csv", index=False)
roll_weekly_nig_bayes_is_df.to_csv(output_path / "nig_bayes_weekly_is.csv", index=False)
final_nig_bayes_train_df.to_csv(output_path / "nig_bayes_train_end_panel.csv", index=False)

import pickle
with open(output_path / "nig_bayes_posterior_results.pkl", "wb") as f:
    pickle.dump(posterior_results_all, f)

print("Saved:")
print(output_path / "nig_bayes_roll_summary.csv")
print(output_path / "nig_bayes_weekly_is.csv")
print(output_path / "nig_bayes_train_end_panel.csv")
print(output_path / "nig_bayes_posterior_results.pkl")

Saved:
c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\nig_bayes_roll_summary.csv
c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\nig_bayes_weekly_is.csv
c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\nig_bayes_train_end_panel.csv
c:\Users\afons\OneDrive\Desktop\ESE\FCS\Merton_NIGbayesian\notebooks_test\..\data\derived\nig_bayes_posterior_results.pkl


In [25]:
from pd_estim_A.models.nig.nig_apath import NIGParams, build_weekly_calendar_from_panel
from pd_estim_A.models.nig.nig_gibbs_weekly_updated import (
    annual_cc_rate_to_weekly,
    invert_assets_on_dates,
)

# failing summary row
row = roll_summary_nig_bayes_df.iloc[0]
gv = str(row["gvkey"])
train_end = pd.Timestamp(row["train_end"])

# exact EM row used
em_row = em_params_df[
    (em_params_df["gvkey"].astype(str) == gv) &
    (pd.to_datetime(em_params_df["train_end"]) == train_end)
].iloc[-1]

p_em = NIGParams(
    alpha=float(em_row["alpha"]),
    beta=float(em_row["beta"]),
    delta=float(em_row["delta"]),
    mu=float(em_row["mu"]),
)

# pull firm panel safely
g = firm_daily[gv].copy()

# if date is in index, bring it out; if already in columns, keep it there
if "date" not in g.columns:
    g = g.reset_index()

# remove duplicate column names if any
g = g.loc[:, ~g.columns.duplicated()].copy()

# standardize date
g["date"] = pd.to_datetime(g["date"], errors="coerce")

# recover the train window directly from the global config
train_start = train_end - pd.DateOffset(years=TRAIN_YEARS) + pd.Timedelta(days=1)

g_train = (
    g.loc[(g["date"] >= train_start) & (g["date"] <= train_end)]
     .dropna(subset=["date", "E", "L", "r"])
     .query("E > 0 and L > 0")
     .sort_values("date")
     .groupby("date", as_index=False)
     .last()
)

weekly_dates = build_weekly_calendar_from_panel(
    g_train[["date", "E", "L", "r"]],
    week_ending=WEEK_ENDING
)

g_train_idx = g_train.set_index("date").sort_index()
g_train_weekly = (
    g_train_idx.reindex(weekly_dates)
    .reset_index()
    .rename(columns={"index": "date"})
)

Ew = g_train_weekly["E"].to_numpy(float)
Lw = g_train_weekly["L"].to_numpy(float)
rw_annual = g_train_weekly["r"].to_numpy(float)
rw_week = annual_cc_rate_to_weekly(rw_annual, weeks_per_year=52)
dates_w = g_train_weekly["date"].to_numpy()

print("EM params used:", p_em)
print("n weekly obs:", len(g_train_weekly))

# test Gibbs convention
try:
    A_path_52, theta_path_52 = invert_assets_on_dates(
        Ew, Lw, rw_week, dates_w, p_em,
        tau_weeks=52.0, U=INVERT_U, n=INVERT_N
    )
    print("tau=52 success")
    print("finite A share:", np.isfinite(A_path_52).mean())
    print("finite theta share:", np.isfinite(theta_path_52).mean())
except Exception as e:
    print("tau=52 failed:", type(e).__name__, str(e))

# test frequentist-weekly convention
try:
    A_path_1, theta_path_1 = invert_assets_on_dates(
        Ew, Lw, rw_week, dates_w, p_em,
        tau_weeks=1.0, U=INVERT_U, n=INVERT_N
    )
    print("tau=1 success")
    print("finite A share:", np.isfinite(A_path_1).mean())
    print("finite theta share:", np.isfinite(theta_path_1).mean())
except Exception as e:
    print("tau=1 failed:", type(e).__name__, str(e))

EM params used: NIGParams(alpha=3.0, beta=0.0036045937960963, delta=0.0006298957681738, mu=0.0014480454547143)
n weekly obs: 105
tau=52 failed: RuntimeError Could not bracket theta root inside admissible domain. Best |g|=1.067e-03 at theta≈-3.003605. Try different starting params or check nig_kappa parameterization.
tau=1 success
finite A share: 1.0
finite theta share: 1.0


In [26]:
from pd_estim_A.models.nig.nig_gibbs_weekly_updated import run_nig_window_for_firm

# pick the same firm/window you tested before
gv = gvkeys_run[0]
w = windows_run[0]

summary_test, weekly_test, result_test = run_nig_window_for_firm(
    firm_daily[gv],
    train_start=w["train_start"],
    train_end=w["train_end"],
    em_params=em_params_df[
        (em_params_df["gvkey"].astype(str) == str(gv)) &
        (pd.to_datetime(em_params_df["train_end"]) == pd.Timestamp(w["train_end"]))
    ].iloc[-1],
    date_col="date",
    week_ending=WEEK_ENDING,
    min_daily_rows=MIN_DAILY_ROWS,
    min_weekly_obs=MIN_WEEKLY_OBS,
    min_weekly_returns=MIN_WEEKLY_RETURNS,
    E_col="E",
    L_col="L",
    r_col="r",
    tau_years=1.0,
    weeks_per_year=52,
    max_iter=100,
    burn_in=20,
    thin=2,
    U=INVERT_U,
    n_int=INVERT_N,
    prior_b0=PRIOR_B0,
    prior_B0=PRIOR_B0_COV,
    prior_hyper=PRIOR_HYPER,
    rng=np.random.default_rng(123),
)

print(summary_test)
print("\nweekly_test shape:", weekly_test.shape)
if result_test is not None:
    print("A_draws shape:", np.asarray(result_test["A_draws"]).shape)
    print("params_weekly shape:", np.asarray(result_test["params_weekly"]).shape)
    print("params_annual shape:", np.asarray(result_test["params_annual"]).shape)

display(weekly_test.head())

{'gvkey': '100022', 'train_start_req': Timestamp('2014-01-01 00:00:00'), 'train_end_req': Timestamp('2015-12-31 00:00:00'), 'train_start_used_daily': Timestamp('2014-01-01 00:00:00'), 'train_end_used_daily': Timestamp('2015-12-31 00:00:00'), 'train_start_used_weekly': Timestamp('2014-01-03 00:00:00'), 'train_end_used_weekly': Timestamp('2015-12-31 00:00:00'), 'ok': False, 'msg': 'no_kept_draws', 'em_alpha': 3.0, 'em_beta': 0.0036045937960963, 'em_delta': 0.0006298957681738, 'em_mu': 0.0014480454547143, 'alpha_post_mean': nan, 'beta_post_mean': nan, 'delta_post_mean': nan, 'mu_post_mean': nan, 'alpha_post_median': nan, 'beta_post_median': nan, 'delta_post_median': nan, 'mu_post_median': nan, 'A_last_mean': nan, 'A_last_median': nan, 'n_daily_train': 522, 'n_weekly_train': 105, 'n_weekly_returns': 104, 'n_keep_requested': 40, 'n_keep_actual': 0, 'n_reject': 100}

weekly_test shape: (0, 0)
A_draws shape: (0, 105)
params_weekly shape: (0, 4)
params_annual shape: (0, 4)


""
